# DPE head aggregation — independent comparison

Run all 30 global weighting-rule and neural-combiner configurations on the same frozen heads. Select a GPU runtime, enable Internet on Kaggle and choose **Run All**. No attachments or code edits are required.

The official training set trains the backbone. Validation images are divided directly into a **head-training split (720 images)** and a **combiner-training split (479 images)**, stratified by bird/background group. The official test set evaluates every configuration independently after all fits are frozen.

Defaults run seeds 0, 1 and 2 and include test evaluation. The notebook downloads official Waterbirds and trains missing artifacts automatically. Keep WORK_DIR to resume completed epochs and reuse completed backbones, heads and model fits. A complete source bundle from this protocol is optional.

This is a project adaptation inspired by Regularized Neural Ensemblers (AutoML 2025). The comparison remains exploratory because the test set has previously informed project research.


In [ ]:
MODE = "auto"       # Prepare/train missing seeds; "cached" requires existing H2 sources.
SOURCE = None       # Optional source bundle or study ZIP from this protocol.
SEEDS = [0, 1, 2]    # Explicitly set [0] for a one-seed pilot.
TRACK = "standard"
EVALUATE_TEST = True
CONFIG_PATH = "configs/dpe_head_reweighting/aggregation_comparison.yaml"
DEVICE = "auto"     # Prefer GPU for new backbones. Neural combiners run on CPU.
WORKERS = None      # Defaults to 2; use 0 if worker processes are unavailable.
DATA_DIR = None     # Discover attached Waterbirds or download automatically.
REPO_PATH = None
WORK_DIR = None
SEARCH_PATHS = None
REPO_URL = "https://github.com/chautanphat/waterbirds-shortcut-mitigation.git"
REPO_REF = "experiment/dpe-neural-combiner"  # Published implementation branch.


In [ ]:
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

working = Path.cwd().resolve()
PLATFORM = "kaggle" if Path("/kaggle/working").is_dir() else "colab" if Path("/content").is_dir() else "local"
def compatible(path):
    marker = path / "src/methods/dpe_head_reweighting/aggregation_study.py"
    return marker.is_file() and "AGGREGATION_STUDY_RUNTIME_VERSION = 6" in marker.read_text()
candidates = [working, *working.parents, working / "waterbirds-shortcut-mitigation"]
REPO = Path(REPO_PATH).expanduser().resolve() if REPO_PATH else next((p for p in candidates if compatible(p)), None)
if WORK_DIR is None:
    base = Path("/kaggle/working") if PLATFORM == "kaggle" else Path("/content") if PLATFORM == "colab" else (REPO / "outputs" if REPO else working)
    current = base / "dpe_head_aggregation_session"
    WORK_DIR = current
WORK_DIR = Path(WORK_DIR).expanduser().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
if REPO is None:
    REPO = WORK_DIR / "repository-aggregation-v6"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
if not compatible(REPO):
    raise FileNotFoundError("Use the updated head-aggregation runtime (version 6); set REPO_PATH to the published implementation branch.")
missing = [package for module, package in [("numpy", "numpy>=1.26,<3"), ("scipy", "scipy>=1.11,<2"),
    ("torch", "torch>=2.6,<3"), ("torchvision", "torchvision>=0.21,<1"),
    ("yaml", "PyYAML>=6,<7"), ("pandas", "pandas>=2.2,<3"),
    ("PIL", "Pillow>=10.3,<13"), ("tqdm", "tqdm")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display, FileLink
from src.methods.dpe_head_reweighting.aggregation_study import run_study
print("Platform:", PLATFORM, "| Repository:", REPO, "| Workspace:", WORK_DIR)


## Fit and freeze every configuration

Head fitting and head-epoch selection use only the head-training split. All global weighting rules and neural combiners use only the combiner-training split. There is no held-out candidate-selection split: every declared configuration is reported separately on test.

The 30 configurations are uniform averaging, 12 power rules, 2 softmax rules, 2 top-k rules, 3 joint-weight fits, and 10 neural configurations. Power, softmax and top-k derive reliability from per-head training WGA. Joint fits minimize worst-group NLL with uniform regularization. Neural models use group-balanced BCE and 400 fixed AdamW steps; both weight-decay settings are separate configurations.

Auto mode trains missing sources. Cached mode requires sources in the current two-split format. All configurations for every planned seed are frozen before test predictions are generated or read. Re-running verifies saved model states, source identities and configuration.


In [ ]:
if SEARCH_PATHS is None:
    SEARCH_PATHS = [REPO / "outputs/dpe_head_aggregation_sources"]
    if PLATFORM == "kaggle":
        SEARCH_PATHS += [Path("/kaggle/input")]
    elif PLATFORM == "colab":
        SEARCH_PATHS += list(Path("/content").glob("*.zip")) + [Path("/content/uploads")]
    else:
        SEARCH_PATHS += list(working.glob("*.zip"))
STUDY = run_study(WORK_DIR, source=SOURCE, search_roots=SEARCH_PATHS, track=TRACK,
                  seeds=SEEDS, config_path=CONFIG_PATH, evaluate_test=EVALUATE_TEST,
                  mode=MODE, device=DEVICE, workers=WORKERS, data_dir=DATA_DIR)
output = Path(STUDY["output_dir"])
for seed in SEEDS:
    selection = json.loads((output / "comparisons" / f"seed{seed}" / "models.json").read_text())
    print("Seed", seed, "protocol:", selection["protocol"])
    print("Frozen configurations:", [r["id"] for r in selection["candidates"]])
    print("Training complementarity (label-aware oracle is not a predictor):", selection["complementarity"])
    print((output / "comparisons" / f"seed{seed}" / "candidate_comparison.csv").read_text())
if STUDY["summary"] is None:
    print("Configurations saved. Test evaluation was not requested. Set EVALUATE_TEST=True and rerun to evaluate.")
else:
    print(json.dumps(STUDY["summary"], indent=2))


## Download models, source bundles and reports

The ZIP updates after each seed. It contains the plan, frozen models, training diagnostics, per-configuration test metrics and reusable source bundles. Sources contain the two-split manifest, frozen heads and combiner-training prediction cache; the test cache is added when evaluation is requested.

Images, backbone checkpoints and raw features stay in WORK_DIR. Preserve that directory to resume interrupted training or generate missing test predictions. A training-only source ZIP cannot reconstruct missing test predictions without the matching raw-feature handoff.


In [ ]:
if "STUDY" in globals():
    archive = Path(STUDY["archive"])
else:
    partial = sorted((WORK_DIR / "studies").glob("*.zip"), key=lambda p: p.stat().st_mtime)
    if not partial:
        raise FileNotFoundError("No seed has frozen yet. Rerun the study cell to resume saved backbone epochs.")
    archive = partial[-1]
previous = Path.cwd()
try:
    os.chdir(working)
    display(FileLink(os.path.relpath(archive, working), result_html_prefix="Download head-aggregation comparison: "))
finally:
    os.chdir(previous)
if PLATFORM == "colab":
    from google.colab import files
    files.download(str(archive))
